# RAG

In [1]:
# Load the environment
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Read the model name
import os
MODEL_NAME = os.environ["GEMINI_MODEL"]
API_KEY = os.environ["GOOGLE_GENERATIVE_AI_API_KEY"]

In [3]:
# Create the LangChain model
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI (model = MODEL_NAME, api_key =API_KEY, temperature = 0)

In [4]:
# Create the document object
from langchain_core.documents import Document
documents = [
    Document(
        page_content="LangChain provides abstractions for building LLM applications.",
        metadata={"source": "langchain.txt"}
    ),
    Document(
        page_content="LangGraph is designed for stateful agent workflows.",
        metadata={"source": "langgraph.txt"}
    ),
]

## Manual RAG

In [5]:
# Create the embedding model
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=API_KEY,
)

In [6]:
# Create the vector store
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding=embeddings)

In [7]:
# Add documents to the vector_store

vector_store.add_documents(documents)

['76154926-cc43-4573-beb4-5eec13ca9c57',
 '3412abe4-8dda-40f9-b0dd-73d32fc7cae1']

In [8]:
# Run the retriever

retriever = vector_store.as_retriever(
    search_kwargs = {"k":2}
)

In [9]:
# Invoke the retriever

query = "What is LangGraph"
retrieved_docs = retriever.invoke(query)

In [10]:
# Create the formatting function
def format_docs(documents: list[Document])->str:
    return "\n\n".join(
        doc.page_content
        for doc in documents
    )

"""
or
def docs_to_text(documents: list[Document])-> str:
    text = ""
    for doc in documents:
        text = text + doc.page_content + "\n\n"
    return text
"""

'\nor\ndef docs_to_text(documents: list[Document])-> str:\n    text = ""\n    for doc in documents:\n        text = text + doc.page_content + "\n\n"\n    return text\n'

In [11]:
context = format_docs(retrieved_docs)
print(context)

LangGraph is designed for stateful agent workflows.

LangChain provides abstractions for building LLM applications.


In [12]:
# Create the prompt

prompt = f"""
Answer the question using the following context.

Context:
{context}

Question:
{query}
"""

In [13]:
# Invoke the llm

response = llm.invoke(prompt)

print(response.content)

[{'type': 'text', 'text': 'Based on the provided context, LangGraph is designed for stateful agent workflows.', 'extras': {'signature': 'Ev4GCvsGARFNMg8gD7FvZoHz2L8Y7Li+O3z5lCSRikj5tD+MqWelf9m9C8BCzKUkrD0XBznVyrQBwiviIQO1KTnpdDoEATKxvTsXoDPNMbe7tXyuhUBXyt+Z3KeI72d4PPqnpqcEEnoCKWoUclk8s1MuyQ3r40YymO/VQx+lydRtKCXu2qMcW8mgabxLgWWZRQaDAeoKlsaOLSdU7VewhQ04qy/EZQeIUvuc4iSupOhUCPvHnNC6RsgkeGvsz3ac1p7zb0gTG2xhKL1ltPZtVjUdU/eT0uGTriL/8BRE4lid7mZQFBjnqTCIhqlTjrHq1lEjpkNj3YA1z/gsm5yrB5g7EU7c2893MzJa6SMbbW6HtTri09isam+CcixeIYM71cTWfex8zO/Ft5Fa6dJMiov51BATX9Ux7W4winUx7Jgl9iZn5W3+EkhyL+DfwQ9MwMvkRTisMA3TbFIKUF6ylKcSC4icxFr+6dtW7mgIjeoo+G/DwsaXB9drBM8ahTdv/yC40uUAeSvD57RGX1O6FkyZHVSfSLe7e61P5bCvCktTZYMn9/hcmBVNPItOOL7unKZy1xcKU2y0sdPl/tZ5C3DtA60IeKRoimQUG6RULoVgHuf+OyGnMeu6WeN9qyMk7ZM2hKAOOZgOtqyez5FgN5X7eypMlZ/oQSWPziqGRBcKBB7Kv45nVSV09I0GJNWrs4WdZqEO9oCCMA7h0g9pQAVMo65P9MviYx/zgxESPGKBGFHhNu46jc2oRJ0X7WLz0KJkl9hmkJJeNRzY45E4DUFcvWOEyH64k98SJzg921qvCCOkiiUAbRXLof29rzZL5fL+WbpNP8ePv7J10ygwBVuIpwZpGld

# RAG Chain with Runnables

In [14]:
query = "What is LangGraph?"

In [15]:
# Convert format_docs, prompt to runnable
from langchain_core.runnables import RunnableLambda

format_docs_runnable = RunnableLambda(format_docs)

In [16]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

retriever_chain = RunnableParallel(
    {
    "context": retriever | format_docs_runnable,
    "query": RunnablePassthrough()
    }
)

In [17]:
# Create prompt
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_template(
"""
Answer the question using the following context.

If the context does not contain enough information to answer the question, 
say that you don't have enough information.

Context:
{context}

Question:
{query}
""")

In [18]:
# Debug 
retrieved = retriever_chain.invoke(query)
print(retrieved['context'])

LangGraph is designed for stateful agent workflows.

LangChain provides abstractions for building LLM applications.


In [19]:
# Create RAG Chain
RAG_Chain = retriever_chain | prompt | llm

In [52]:
RAG_response = RAG_Chain.invoke(query)

In [53]:
print(RAG_response.content)

[{'type': 'text', 'text': 'Based on the provided context, LangGraph is designed for stateful agent workflows.', 'extras': {'signature': 'EsIHCr8HARFNMg8ZtRXf6i1VMQNaAjiM9615oDpIgFck/vyKCWJNORIqDXlpSX2CWuJ8MzCdim9Df71IYi+dAxd3FfH40vXXzcsfGdp+OeK+udAfE236oH6e4Bj3hGTZRpuGE449Kx21wBArbMoTiEeCfmXrwvcgtZ75I3WlI1QAjoVVvTMQ+WV7AzlFiniZsmdGzMdIa7DZIOl6omEsU8090p3tOv4pvche7QAjLIQOIS1NNZPL6K7rs5rz9OJJ5ea29MSEYW3X4Mo5fw7vvw7r7QfSHaTEzsHFTTzCwJosn3vdlLMY/IkrTLTa3sLg4Akz6+zCMQEZfpF7N/8v7EFtr8T2VA/rRTPkFQl1fmZO3us4G6MJ0/bVrP6+Dde6mUwh1fajKCY6vL85TK4BTPZWY2OGptnb1uBgAaEFezJF96z/uO6MQOGv/nGs46sPJALgCakEtSL0s+QiwSu1RIbDNG6vFn9Qdx2VbQ5UaONq6/jelgrfAnbaHODNHtLADAsCg41AZmcfGdAtu8ZpSP56sebIvboEUDx82aPJPvo4AssP/3S9yB1y/YScqbgKdDmtBmf4jeHk9x+T5CxTkJbSASj/70sWL5Tdbywr34Giehy0SjeZHeI4mEKYl5Dpn5RmdolOqAy+rM1LgtioZQhEc8Ae4ViqV9V7hQMif+C/YENAxnn520VVsGdyEH86QSVuEMbGVKfPDG5AaIG0kSDEEUseZ8OXdq8/qrnn3fHJ3lzA84bMQszHXCXo1pafLnD5OUnCt/X9fiovQehwTa0vwwjON6lRivee3W6PcZI2f0bb+LoRQ6nXk8d+zZB93Q5QNCqCS3Q/AGiYI2AHMcXxBPfg2UQ

RAG 2 major stages:

```text
                    RAG
                     │
          ┌──────────┴──────────┐
          ▼                     ▼
     Retrieval stage       Generation stage
          │                     │
     Find relevant docs    Generate answer
```

In [22]:
response = RAG_Chain.invoke("What is the newest AMD GPU chip")

In [23]:
print(response.content)

[{'type': 'text', 'text': "I don't have enough information.", 'extras': {'signature': 'EtMHCtAHARFNMg+g0171AeZzaKBVMK8++b9zuILXErnAWIa8dReXkCXEIRwriytN63tZS3hwD7rGFkTczv1VkxSYrv8SPBi3zm0wGjGntmcZTEBrl6RakpiUGcRIDgNPR3d6LkJJbs2lhyHk5EFmhzKnRe8p5QDK+qnad0NFxJQWzPvjOA2tbpROSSH9+fEIakL8JhhXIvmuPg2wFyLAvbpGKSgbdTqma9PNAVzG0C4lars0kLC3bnPTu1TaaktxhV7nh/JadbwwZ2GyocuU5MZYiAzVXRt5IFKCJYFjcTZOFu4C2zcYwexGNh/q+Wa3ZiBmYBbWOnA5CrmkUgZ43l1eJB3Tw4D+wC+6+e4g58P8dvKd8TgLnjwySDOQA/P/EbE5o99OXVzozruWmbEOHg6G8rt0faaFxHO4YNkiHPiXJWs6rEawOe3MjeT6cHWA1Hn30WuIFRFBDc6wXYyx/p1IF3kGevQYuAfr/tQy7JZovgngJ6ymZuH1we64QiwV5nN5QZn5l0nrpaS4QKtWeo62arnFROXs5+TKf6xH8E8VDpLOWel1WIYvPgPI6wGr57M0CAeQmVLGRndCGLuUwQdvBgK6PsVahUAf8bjBWKUuPxsAv1vo0uxHOvPd1kiRrdGMMWpLOQTo7hI7fCzgWOXSXPZ10j3cNyrln6HRCEm86DIqZu3jVSh/HH53elKmAzVbSdC9mUZDnBwd4jDLWor3s78/iAWKYeded9D524ZkWLB1AXeV6URTEva2VotY/Z01JnyJPzxHuhSyu9FGuzfFWqy2SPgUosg32AJevyVd13+NJg2VJ7NpMqqId7s/MGxuaC7UJqPadYGSkqcH40sUN4zLNfcerOx9DIdxm3ofb32OYXKe1kUefAXPpp+QYWBO3r5TMSKUVzHz/

## Advanced RAG pipeline

Work Flow:
```text
Conversation history
        ↓
Query Rewriter
        ↓
Standalone query
        ↓
Retriever (MMR)
        ↓
Relevant documents
        ↓
Formatted context
        ↓
History + current question + context
        ↓
LLM
        ↓
Answer
```

In [24]:
# Create the document objects
from langchain_core.documents import Document

documents = [
    Document(
        page_content="LangChain provides abstractions for building LLM applications.",
        metadata={"source": "langchain.txt"}
    ),
    Document(
        page_content="LangGraph is designed for stateful agent workflows.",
        metadata={"source": "langgraph.txt"}
    ),
    Document(
        page_content="Retrievers are components that return relevant documents for a given query.",
        metadata={"source": "retrievers.txt"}
    ),
    Document(
        page_content="Vector stores are used to store and search vector representations of documents.",
        metadata={"source": "vector_stores.txt"}
    ),
    Document(
        page_content="Embeddings represent text as numerical vectors that capture semantic relationships.",
        metadata={"source": "embeddings.txt"}
    ),
    Document(
        page_content="RAG combines information retrieval with language generation to provide context to an LLM.",
        metadata={"source": "rag.txt"}
    ),
    Document(
        page_content="LCEL allows LangChain components to be composed into executable pipelines using the pipe operator.",
        metadata={"source": "lcel.txt"}
    ),
    Document(
        page_content="RunnableParallel allows multiple Runnable components to execute using the same input.",
        metadata={"source": "runnable_parallel.txt"}
    ),
    Document(
        page_content="RunnablePassthrough forwards the original input without modifying it.",
        metadata={"source": "runnable_passthrough.txt"}
    ),
    Document(
        page_content="RunnableLambda converts a Python function into a Runnable component.",
        metadata={"source": "runnable_lambda.txt"}
    ),
    Document(
        page_content="Prompt templates provide a reusable structure for constructing prompts dynamically.",
        metadata={"source": "prompt_templates.txt"}
    ),
    Document(
        page_content="Structured output allows an LLM response to follow a predefined schema.",
        metadata={"source": "structured_output.txt"}
    ),
    Document(
        page_content="Pydantic models can be used to define and validate structured data returned by an LLM.",
        metadata={"source": "pydantic.txt"}
    ),
    Document(
        page_content="Tool calling allows an LLM to request that an external function or system be used.",
        metadata={"source": "tool_calling.txt"}
    ),
    Document(
        page_content="A tool call contains the name of the requested tool and the arguments generated by the LLM.",
        metadata={"source": "tool_calls.txt"}
    ),
    Document(
        page_content="The application executes a tool after receiving a tool call from the LLM.",
        metadata={"source": "tool_execution.txt"}
    ),
    Document(
        page_content="Tool descriptions help an LLM understand when a tool should be selected and how it should be used.",
        metadata={"source": "tool_descriptions.txt"}
    ),
    Document(
        page_content="Similarity search retrieves documents whose vector representations are close to the query vector.",
        metadata={"source": "similarity_search.txt"}
    ),
    Document(
        page_content="Maximum Marginal Relevance can improve retrieval diversity by balancing relevance and similarity between retrieved documents.",
        metadata={"source": "mmr.txt"}
    ),
    Document(
        page_content="Metadata filtering allows retrieval results to be restricted using attributes associated with documents.",
        metadata={"source": "metadata_filtering.txt"}
    ),
]

In [25]:
# Add documents to the vector_store

vector_store.add_documents(documents)

['df7d359b-4165-4afe-9299-087ba0048095',
 '8fa9d910-0a31-4166-80a1-2ccc33fcb6c9',
 'ce5c55a4-9bf4-48c9-ae11-5266a570325c',
 'c67683fa-aa17-4560-a0bb-d671002d6e7c',
 '337ed967-b2df-45cf-a04b-c3edce01b223',
 'b5c65e9f-c91b-4e03-aac5-33c1bbaaa5f6',
 '909cd9a3-ee4b-45b4-b8e4-42d0f9adc219',
 'c1bc20a8-ad85-440a-89d6-5641830a4d06',
 '10c014d8-9409-4a70-9fd2-a552a57d1561',
 'd367256e-ffb5-458c-8687-a428a37a71b5',
 'b1a1d858-894f-466b-aec7-344465e27898',
 'acc48099-c46a-4570-aed7-951af4c0023e',
 'f742dc94-8d9e-4385-86da-38a84f7af612',
 '080c673f-74e6-431a-9ff6-99faacf4a417',
 '45961115-d654-4b2c-aaae-b8ac4f95f8c2',
 '0c814789-a445-4b4a-8e57-c5e782c98985',
 'f1029a03-7f2b-43b3-94d0-376bc1509a56',
 'd87118cd-ab18-4dd7-a7d3-a150d1fd40c6',
 '4156f1f7-e2e4-4765-9e1f-25b0b5eec274',
 '4ed1bb7d-3bfb-4b3b-aac2-cdcb6f52d762']

In [26]:
# Create the retriever

retriever = vector_store.as_retriever(
    search_type ="mmr",
    search_kwargs = {
        "k":3,
        "fetch_k": 10
        }
)

In [27]:
# Create the formatting function
def format_docs(documents: list[Document])->str:
    docs = "\n\n".join(
        f"Page content: {doc.page_content}\n Page_source: {doc.metadata}"
        for doc in documents
        )
    return docs
    


In [28]:
# Convert the format function to runnable
format_docs_runnable = RunnableLambda(format_docs)

In [54]:
# Create history
from langchain_core.messages import HumanMessage, AIMessage

history = [
    HumanMessage(content=query),
    AIMessage(content=RAG_response.content[0]['text'])
]



In [55]:
print(history)

[HumanMessage(content='What is LangGraph?', additional_kwargs={}, response_metadata={}), AIMessage(content='Based on the provided context, LangGraph is designed for stateful agent workflows.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [31]:
# Create history
"""from langchain_core.messages import HumanMessage, AIMessage

def history (query, previous_query, previous_response):
    history = [
        HumanMessage(content=query),
        AIMessage(content=response.content[0]['text'])
    ]
    return(query, history)

"""

"from langchain_core.messages import HumanMessage, AIMessage\n\ndef history (query, previous_query, previous_response):\n    history = [\n        HumanMessage(content=query),\n        AIMessage(content=response.content[0]['text'])\n    ]\n    return(query, history)\n\n"

In [32]:
retriever_chain = RunnableParallel({
    "context": retriever | format_docs_runnable,
    "query": RunnablePassthrough(),

})

In [33]:
query2 = "what is the use case of it?"

In [34]:
# Query rewriter 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
query_rewrite_prompt = ChatPromptTemplate([
    (
        "system",
        "Rewrite the user's latest question into a standalone question "
        "that can be understood without the conversation history. "
        ),
        MessagesPlaceholder("history"),
        ("human", "{query}")
])

In [35]:
from langchain_core.output_parsers import StrOutputParser
query_rewriter = query_rewrite_prompt | llm 
# query_rewriter = query_rewrite_prompt | llm | StrOutputParser()

In [36]:
rewritten_query = query_rewriter.invoke({
    "history": history,
    "query": query2
})


In [37]:
rewritten_query_str = rewritten_query.content[0]["text"]

In [38]:
retrieved_docs = retriever_chain.invoke(rewritten_query_str)
print(retrieved_docs)

{'context': "Page content: LangGraph is designed for stateful agent workflows.\n Page_source: {'source': 'langgraph.txt'}\n\nPage content: Embeddings represent text as numerical vectors that capture semantic relationships.\n Page_source: {'source': 'embeddings.txt'}\n\nPage content: The application executes a tool after receiving a tool call from the LLM.\n Page_source: {'source': 'tool_execution.txt'}", 'query': 'What are the use cases of LangGraph?'}


In [39]:
# Create the prompt
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer the question using the provided context. If the context does not contain enough information to answer the question, say that you don't have enough information."),
        MessagesPlaceholder("history"),
        ("human", "Question: {query}, \nContext: {context}"),
    ]
)

In [40]:
prompt__history = prompt.invoke(
    {
        "history":history,
        "context":retrieved_docs,
        "query":query2
    }
)

In [41]:
RAG_response =  llm.invoke(prompt__history)

In [42]:
print(RAG_response)

content=[{'type': 'text', 'text': 'Based on the provided context, LangGraph is designed for stateful agent workflows.', 'extras': {'signature': 'EoYKCoMKARFNMg+MU2/ANcPEffAkzMK1QCRpbdBf4APCQxh/CIjJKRdClco1FHBdKfbPW+056gVKvWZ06SU3+ChHuEYcmIW4QodBazjK1G1kyIsI3QT9Sgd20VNaXFvrIUDXufocbyf2IyJVxDBHY0z4rETf1GEgC3HiOj9nSxWwLzURyz6ukasQSi5T2I0B8VNOtGpUK6cZ4m+ugv+W1j28C9g5qr7RzML+NVINOJHAro1EQrB4CokXEUNoLZFvD0FpX8B0R6GP/bGALl6+8ffWoYEdC5w+7IOlPIZXI+MafWW0jnUhQDlW1AosCsq0CHe8K2nOA21TRcKFZG/q+GJpIaUkkgqJ85ChVcZlOIi1hca65bFZRLP0UhVRpvNxyTEt+VZsQ6zco8GQOuHK2/BgDnVt8k5AyiTnUWrfG4yW464EJD3rG8vXKSuSOcgFML+3XJZbcqLP6Mhaw1g2/V8SdFsG9Aot31A9C3VQhdkDcAqZrAzw05MT0QxbKd9HE04XRxxiAz/Ts0ZjAdWInzSZmffZdQEdHnOJSxf4tBUhV0N57C1Xn2Q0MUpuNGFFOlXFsYdOfyxR/VJN/rjdAumne9UyXq67pPJU/CgodmhCsYNf82Kh6xZ1zZd0VB6lFH9ODNo7uRfe9DpTFJeesSkgzIgsBzbxg3piUhEZWn13K8es+Ge+MZxyMNtQARc5Wl6Rl/NYlzO8LYfZQ26lAcexQpiLa2UTzzaUxHfNugWWyzrhOLE0hUYstfxRHpXPGklHIcKKclIyc36q3c3Gnj5BIkVheoNKRXRJQ1cfSgNZRnbXo7pvGFkI+hUhoWtER9pnxRZAr7B7K26bh78mkIK

## Advanced RAG pipeline improvement

In [ ]:
query2 = "what is the use case of it?"

In [ ]:
# Create the conversation input
# Wont be used any further but for illustration 
conversation_input = RunnableParallel({
    "history": RunnablePassthrough(),
    "query": RunnablePassthrough(),
})

In [ ]:
# Invoke the conversation input
conversation_input.invoke({
    "history": history,
    "query": query2
}) 

{'history': {'history': [HumanMessage(content='What is LangGraph?', additional_kwargs={}, response_metadata={}),
   AIMessage(content="I don't have enough information.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
  'query': 'what is the use case of it?'},
 'query': {'history': [HumanMessage(content='What is LangGraph?', additional_kwargs={}, response_metadata={}),
   AIMessage(content="I don't have enough information.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
  'query': 'what is the use case of it?'}}

In [ ]:
# Create conversation context

conversation_context = RunnableParallel({
    "history": RunnableLambda(lambda x: x ["history"]),
    "query": RunnableLambda(lambda x: x ["query"]),
    "rewritten_query": query_rewriter | StrOutputParser()
})

In [74]:
conversation_context_result = conversation_context.invoke({
    "history": history,
    "query": query2
})

print(conversation_context_result)

{'history': [HumanMessage(content='What is LangGraph?', additional_kwargs={}, response_metadata={}), AIMessage(content='Based on the provided context, LangGraph is designed for stateful agent workflows.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'query': 'what is the use case of it?', 'rewritten_query': 'What are the use cases of LangGraph?'}


In [ ]:
# Get the rewritten query 

rewritten_query = conversation_context_result['rewritten_query']
rewritten_query

'What are the use cases of LangGraph?'

In [76]:
print(type(rewritten_query))

<class 'langchain_core.messages.base.TextAccessor'>


In [ ]:
# Invoke the retriever chain

retrieved_docs = retriever_chain.invoke(rewritten_query)
print(retrieved_docs)

{'context': "Page content: LangGraph is designed for stateful agent workflows.\n Page_source: {'source': 'langgraph.txt'}\n\nPage content: Embeddings represent text as numerical vectors that capture semantic relationships.\n Page_source: {'source': 'embeddings.txt'}\n\nPage content: The application executes a tool after receiving a tool call from the LLM.\n Page_source: {'source': 'tool_execution.txt'}", 'query': 'What are the use cases of LangGraph?'}


In [ ]:
# Create the RAG chain

RAG_chain =  prompt | llm

In [ ]:
RAG_response =  RAG_chain.invoke({
    "history": history,
    "context": retrieved_docs["context"],
    "query": query2
})

In [ ]:
str_RAG_response = StrOutputParser().invoke(RAG_response)
print(str_RAG_response)

Based on the provided context, LangGraph is used for stateful agent workflows.
